# Benchmarks and Lexicons Load

## Lexicon Data Source

 **The NRC Emotion Intensity Lexicon** (NRC-EIL)
 - Real-valued scores of intensity for eight basic emotions (anger, anticipation, disgust, fear, joy, sadness, surprise, and trust)
 - https://saifmohammad.com/WebPages/AffectIntensity.htm 
 - Raw data dowloaded to `NRC-Emotion-Intensity-Lexicon-v1.txt` in `data_in/`
 - Validated and loaded into `NRC-Emotion-Intensity-Lexicon-v1.loaded-YYYY-MM-DD.json`

 Some parsing and storage rules for the source file:
 - Three tab-separated columns — `word`, `emotion`, `emotion-intensity-score`
 - Assumes an affect representation of `EKMAN6`, `PLUTCHIK8`


In [1]:
# Get the root path and data paths
#

import json
from datetime import UTC, datetime
from pathlib import Path


def repo_root(marker: str = "uv.lock") -> Path:
    """Nearest ancestor of the working directory containing *marker*."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(f"No {marker} found above {start}")

DATA_IN = repo_root() / "data_in"
SOURCE_TSV = DATA_IN / "NRC-Emotion-Intensity-Lexicon-v1.txt"

PREPARED_ON = datetime.now(UTC).date().isoformat()
ARTEFACT = DATA_IN / f"{SOURCE_TSV.stem}.loaded-{PREPARED_ON}.json"

print(f'source   : "{SOURCE_TSV}"  exists={SOURCE_TSV.exists()}')
print(f'artefact : "{ARTEFACT}"  exists={ARTEFACT.exists()}')


source   : "/Users/stuartgow/PhD Project/Repo asa_research_prototype/data_in/NRC-Emotion-Intensity-Lexicon-v1.txt"  exists=True
artefact : "/Users/stuartgow/PhD Project/Repo asa_research_prototype/data_in/NRC-Emotion-Intensity-Lexicon-v1.loaded-2026-08-11.json"  exists=True


In [2]:
# Parse the source file
#

entries: dict[str, dict[str, float]] = {}
zero_scored, header = 0, None

rows = SOURCE_TSV.read_text(encoding="utf-8").splitlines()
for number, row in enumerate(rows, start=1):
    # Skip blank rows
    if not row.strip():
        continue
    # Get valid rows
    try:
        word, emotion, score = row.split("\t")
    except ValueError:
        raise ValueError(f"{SOURCE_TSV.name}:{number}: expected three tab-separated fields, got {row!r}") from None 
    # Check magnitude field
    try:
        magnitude = float(score)
    except ValueError:
        raise ValueError(f"{SOURCE_TSV.name}:{number}: {score!r} is not a number") from None
    # Add row to entries
    if emotion not in entries:
        entries[emotion] = {}
    entries[emotion][word] = magnitude
    if magnitude == 0.0:
        zero_scored += 1
    
print(f"{len(rows):,} rows -> {len(entries)} emotions, "
      f"{sum(len(v) for v in entries.values()):,} word-emotion pairs "
      f"({zero_scored:,} zero-scored rows)")
print(f"emotions: {sorted(entries)}")


9,829 rows -> 8 emotions, 9,829 word-emotion pairs (5 zero-scored rows)
emotions: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']


In [3]:
# Store the resulting entries
#

# Attributes
prepared = {
    "id": "nrc-eil",                                    # reaches every record's `source`
    "lexicon": SOURCE_TSV.name,                         # for a human reading the file
    "prepared_by": "notebooks/eval_data_loads.ipynb",
    "prepared_on": PREPARED_ON,                         # survives a rename; the filename does not
    "entries": entries,
}

ARTEFACT.write_text(json.dumps(prepared, indent=1, sort_keys=True), encoding="utf-8")
print(f'wrote "{ARTEFACT}"  ({ARTEFACT.stat().st_size / 1024:.0f} KiB)')

wrote "/Users/stuartgow/PhD Project/Repo asa_research_prototype/data_in/NRC-Emotion-Intensity-Lexicon-v1.loaded-2026-08-11.json"  (207 KiB)


In [4]:
# Test that this loads properly
#

import json

from asa.core.representations import EKMAN6, PLUTCHIK8
from asa.perception.nrc_eil import load_tables

# Attributes
print("Attributes")
written = json.loads(ARTEFACT.read_text(encoding="utf-8"))
for key, value in sorted(written.items()):
    if not isinstance(value, (dict, list)):
        print(f"{key:12}: {value}")
print("id present  :", "id" in written)
print(f"{'entries':12}: {len(written.get('entries', {}))} emotions")
print("emotions    :", sorted(entries))

# Test the loader
lex = load_tables(ARTEFACT, [PLUTCHIK8, EKMAN6])
print("Loader")
print(f"source      : {lex.source}")
for rep_id, table in lex.tables.items():
    print(f"{rep_id:<12} {len(table)} axes, {sum(len(w) for w in table.values()):,} words")

Attributes
id          : nrc-eil
lexicon     : NRC-Emotion-Intensity-Lexicon-v1.txt
prepared_by : notebooks/eval_data_loads.ipynb
prepared_on : 2026-08-11
id present  : True
entries     : 8 emotions
emotions    : ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']
Loader
source      : nrc-eil
plutchik8/1  8 axes, 9,824 words
ekman6/1     6 axes, 7,472 words


In [5]:
# Quick look at loaded lexicon
#

import pandas as pd

df = pd.DataFrame(
    [(rep_id, str(axis), word, score)
     for rep_id, table in lex.tables.items()
     for axis, words in table.items()
     for word, score in words.items()],
    columns=["representation", "axis", "word", "score"],
)
display(df.shape)
display(df.head(10))

wide = (df[df.representation == "plutchik8/1"]
        .pivot(index="word", columns="axis", values="score"))
wide.sort_values("happiness", ascending=False).head(20)

(17296, 4)

,representation,axis,word,score
0,plutchik8/1,anger,abandoned,0.222
1,plutchik8/1,anger,abandonment,0.438
2,plutchik8/1,anger,abhor,0.816
3,plutchik8/1,anger,abhorrent,0.875
4,plutchik8/1,anger,abolish,0.485
5,plutchik8/1,anger,abomination,0.844
6,plutchik8/1,anger,abuse,0.812
7,plutchik8/1,anger,accursed,0.588
8,plutchik8/1,anger,accusation,0.510
9,plutchik8/1,anger,accused,0.641


axis,anger,anticipation,disgust,fear,happiness,sadness,surprise,trust
word,,,,,,,,
happiest,NaN,NaN,NaN,NaN,0.986,NaN,NaN,NaN
happiness,NaN,0.641,NaN,NaN,0.984,NaN,NaN,NaN
bliss,NaN,NaN,NaN,NaN,0.971,NaN,NaN,NaN
celebrating,NaN,0.672,NaN,NaN,0.970,NaN,NaN,NaN
jubilant,NaN,NaN,NaN,NaN,0.969,NaN,0.578,0.492
ecstatic,NaN,0.688,NaN,NaN,0.954,NaN,0.562,NaN
elation,NaN,NaN,NaN,NaN,0.944,NaN,NaN,NaN
bestdayever,NaN,NaN,NaN,NaN,0.938,NaN,NaN,NaN
beaming,NaN,0.523,NaN,NaN,0.938,NaN,NaN,NaN


In [6]:
# per-word emotion counts — the Counter you ran, post-filter
display(wide.notna().sum(axis=1).value_counts().sort_index())

# score distribution per axis — where the mass sits, and how thin the low tail is
display(df[df.representation == "plutchik8/1"].groupby("axis").score.describe())

# how much survives below 0.1
display(df.query("representation == 'plutchik8/1' and score < 0.1").shape[0])

1    3677
2    1094
3     647
4     373
5      81
6      14
7       3
8       2
Name: count, dtype: int64

,count,mean,std,min,25%,50%,75%,max
axis,,,,,,,,
anger,1480.0,0.499961,0.206924,0.011,0.35075,0.500,0.641,0.964
anticipation,862.0,0.500283,0.122763,0.148,0.41400,0.500,0.586,0.859
disgust,1092.0,0.500697,0.183824,0.039,0.37500,0.500,0.633,0.953
fear,1763.0,0.499044,0.208910,0.016,0.34400,0.500,0.656,0.984
happiness,1264.0,0.501782,0.216414,0.016,0.32800,0.500,0.672,0.986
sadness,1290.0,0.503548,0.204819,0.009,0.35900,0.500,0.656,0.969
surprise,583.0,0.500422,0.197647,0.055,0.35200,0.508,0.648,0.930
trust,1490.0,0.510451,0.137962,0.117,0.42200,0.516,0.609,0.906


135

In [7]:
from collections import defaultdict

by_word = defaultdict(list)
for line in rows:
    parts = line.split("\t")
    if len(parts) != 3:
        continue
    try:
        by_word[parts[0]].append(float(parts[2]))
    except ValueError:
        continue

print("by_word entries    :", len(by_word))
print("raw distinct words :", len(by_word))
print("wide rows          :", wide.shape[0])
print("missing from wide  :", len(set(by_word) - set(wide.index)))
print("zero rows in file  :", sum(v == 0.0 for vals in by_word.values() for v in vals))
print("words with a zero  :", sum(0.0 in vals for vals in by_word.values()))
print("words all-zero     :", sum(max(vals) == 0.0 for vals in by_word.values()))
vanished = sorted(w for w, vals in by_word.items() if max(vals) == 0.0)
print("vanished.          :", len(vanished), vanished)

by_word entries    : 5891
raw distinct words : 5891
wide rows          : 5891
missing from wide  : 0
zero rows in file  : 5
words with a zero  : 5
words all-zero     : 0
vanished.          : 0 []


In [ ]:
# Lookup a word to get the affectvector
#

from asa.core.affect import Utterance
from asa.perception.decode_keyword import KeywordDecoder

dec = KeywordDecoder(PLUTCHIK8, lex.tables["plutchik8/1"], lexicon=lex.source)
obs = await dec.decode(Utterance(text="angelic", source="notebook"))

print(obs)

AffectEvidence(target=<Target.OTHER: 'other'>, affect=AffectVector(representation='plutchik8/1', values={<EightEmotions.ANGER: 'anger'>: 0.0, <EightEmotions.ANTICIPATION: 'anticipation'>: 0.0, <EightEmotions.DISGUST: 'disgust'>: 0.0, <EightEmotions.FEAR: 'fear'>: 0.0, <EightEmotions.HAPPINESS: 'happiness'>: 0.688, <EightEmotions.SADNESS: 'sadness'>: 0.0, <EightEmotions.SURPRISE: 'surprise'>: 0.0, <EightEmotions.TRUST: 'trust'>: 0.586}), confidence=None, source='decoder:rule:nrc-eil', rationale='matched: happiness=angelic, trust=angelic', computed_from=None, of_input='6095ee20750f', at=datetime.datetime(2026, 8, 11, 23, 10, 49, 553573, tzinfo=datetime.timezone.utc), schema='evidence/1')
